## 1. cxr jpg metadata

In [1]:
import pandas as pd

# Load gzipped CSV metadata file
metadata_path = "/hpc/group/kamaleswaranlab/mimic_cxr/mimic_cxr_jpg/mimic-cxr-2.0.0-metadata.csv.gz"
df = pd.read_csv(metadata_path)

# 1. Inspect format: columns, dtypes, shape
print("=" * 60)
print("[Metadata Structure]")
print("=" * 60)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print("\nColumn names and dtypes:")
print(df.dtypes)
print("\nInfo:")
print(df.info())

[Metadata Structure]
Rows: 377,110
Columns: 12

Column names and dtypes:
dicom_id                                       object
subject_id                                      int64
study_id                                        int64
PerformedProcedureStepDescription              object
ViewPosition                                   object
Rows                                            int64
Columns                                         int64
StudyDate                                       int64
StudyTime                                     float64
ProcedureCodeSequence_CodeMeaning              object
ViewCodeSequence_CodeMeaning                   object
PatientOrientationCodeSequence_CodeMeaning     object
dtype: object

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 377110 entries, 0 to 377109
Data columns (total 12 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  

In [2]:
# 2. Sample rows
print("=" * 60)
print("[First 5 rows]")
print("=" * 60)
display(df.head())

print("\n" + "=" * 60)
print("[Random 5 rows]")
print("=" * 60)
display(df.sample(5, random_state=42))

print("\n" + "=" * 60)
print("[Column summary stats]")
print("=" * 60)
print(df.describe(include='all'))

[First 5 rows]


,dicom_id,subject_id,study_id,PerformedProcedureStepDescription,ViewPosition,Rows,Columns,StudyDate,StudyTime,ProcedureCodeSequence_CodeMeaning,ViewCodeSequence_CodeMeaning,PatientOrientationCodeSequence_CodeMeaning
0,02aa804e-bde0afdd-112c0b34-7bc16630-4e384014,10000032,50414267,CHEST (PA AND LAT),PA,3056,2544,21800506,213014.531,CHEST (PA AND LAT),postero-anterior,Erect
1,174413ec-4ec4c1f7-34ea26b7-c5f994f8-79ef1962,10000032,50414267,CHEST (PA AND LAT),LATERAL,3056,2544,21800506,213014.531,CHEST (PA AND LAT),lateral,Erect
2,2a2277a9-b0ded155-c0de8eb9-c124d10e-82c5caab,10000032,53189527,CHEST (PA AND LAT),PA,3056,2544,21800626,165500.312,CHEST (PA AND LAT),postero-anterior,Erect
3,e084de3b-be89b11e-20fe3f9f-9c8d8dfe-4cfd202c,10000032,53189527,CHEST (PA AND LAT),LATERAL,3056,2544,21800626,165500.312,CHEST (PA AND LAT),lateral,Erect
4,68b5c4b1-227d0485-9cc38c3f-7b84ab51-4b472714,10000032,53911762,CHEST (PORTABLE AP),AP,2705,2539,21800723,80556.875,CHEST (PORTABLE AP),antero-posterior,NaN



[Random 5 rows]


,dicom_id,subject_id,study_id,PerformedProcedureStepDescription,ViewPosition,Rows,Columns,StudyDate,StudyTime,ProcedureCodeSequence_CodeMeaning,ViewCodeSequence_CodeMeaning,PatientOrientationCodeSequence_CodeMeaning
197915,1cf35d28-1d854270-9291fba8-3efeca3c-8ecd0055,15251527,59708228,CHEST (PA AND LAT),PA,2544,3056,21781217,213825.500,CHEST (PA AND LAT),postero-anterior,Erect
297178,6b1ec0be-f88f302a-48bb152e-db813f61-d0bccf41,17890643,55438144,CHEST (PA AND LAT),AP,2544,3056,21621006,233421.718,CHEST (PA AND LAT),antero-posterior,Erect
195157,834becd5-de7cc7de-4f498574-c769c7a2-c31ee980,15187487,50734654,CHEST (PA AND LAT),LATERAL,3056,2544,21560530,110259.281,CHEST (PA AND LAT),lateral,Erect
146520,461672b1-db71805f-4ac56ef9-a9afc131-da3beded,13903940,59023791,CHEST (PORTABLE AP),AP,2544,3056,21590616,63328.437,CHEST (PORTABLE AP),antero-posterior,Erect
345934,70fb3947-947b0c43-a9696d82-5689474a-e87b2afc,19169852,50161519,CHEST (PORTABLE AP),AP,2544,3056,21850201,163702.609,CHEST (PORTABLE AP),NaN,NaN



[Column summary stats]


                                            dicom_id    subject_id  \
count                                         377110  3.771100e+05   
unique                                        377110           NaN   
top     1a1fe7e3-cbac5d93-b339aeda-86bb86b5-4f31e82e           NaN   
freq                                               1           NaN   
mean                                             NaN  1.500806e+07   
std                                              NaN  2.879640e+06   
min                                              NaN  1.000003e+07   
25%                                              NaN  1.250332e+07   
50%                                              NaN  1.501928e+07   
75%                                              NaN  1.749247e+07   
max                                              NaN  1.999999e+07   

            study_id PerformedProcedureStepDescription ViewPosition  \
count   3.771100e+05                            341598       361341   
unique           

## 2. iv admission ID supertables

In [10]:
import pickle
from pathlib import Path
import subprocess
import tempfile
import pandas as pd

# Supertables directory
supertables_dir = Path("/hpc/group/kamaleswaranlab/mimic_iv/sepy_output/mimic-supertables/Supertables")
sample_pkl = supertables_dir / "24562745.pkl"
print(f"Sample file: {sample_pkl.name}")

# If MedTVT-R1 numpy raises RuntimeError, load via base env subprocess and save as CSV
def load_via_subprocess(pkl_path):
    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
        csv_path = tmp.name
    loader = f'''
import pickle,sys,pandas as pd
pkl,csv=sys.argv[1],sys.argv[2]
with open(pkl,"rb") as f: d=pickle.load(f)
pd.DataFrame(d).to_csv(csv,index=True)
'''
    r = subprocess.run(["conda", "run", "-n", "base", "python", "-c", loader, str(pkl_path), csv_path],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"subprocess failed: {r.stderr}\nRun: conda activate base && pip install pandas")
    return pd.read_csv(csv_path, index_col=0)

try:
    with open(sample_pkl, 'rb') as f:
        data = pickle.load(f)
except RuntimeError as e:
    if "empty_like" in str(e) or "numpy" in str(e).lower():
        print("MedTVT-R1 numpy has known bug, falling back to base env...")
        data = load_via_subprocess(sample_pkl)
    else:
        raise

print("\n" + "=" * 60)
print("[pkl data structure]")
print("=" * 60)
print(f"Type: {type(data).__name__}")

if isinstance(data, pd.DataFrame):
    # pkl contains single-patient supertable: rows=timepoints, cols=variables
    print(f"Shape: {data.shape} (rows=timepoints, cols=variables)")
    print(f"Columns: {len(data.columns)}")
    print("\nColumn names (first 30):", list(data.columns)[:30])
    print("\n[First 5 rows]")
    display(data.head())
    print("\n[Column dtype summary]")
    print(data.dtypes.value_counts())
else:
    # If dict structure
    print(f"Keys: {list(data.keys())}")
    for k in list(data.keys())[:10]:
        v = data[k]
        t = type(v).__name__
        s = f"shape={v.shape}" if hasattr(v, 'shape') else f"len={len(v)}"
        print(f"  {k}: {t} {s}")

Sample file: 24562745.pkl
MedTVT-R1 numpy has known bug, falling back to base env...



[pkl data structure]
Type: DataFrame
Shape: (185, 165) (rows=timepoints, cols=variables)
Columns: 165

Column names (first 30): ['age', 'gender', 'race', 'cci9', 'cci10', 'anion_gap', 'base_excess', 'bicarb_(hco3)', 'blood_urea_nitrogen_(bun)', 'calcium', 'calcium_ionized', 'chloride', 'creatinine', 'gfr', 'glucose', 'magnesium', 'osmolarity', 'phosphorus', 'potassium', 'sodium', 'haptoglobin', 'hematocrit', 'hemoglobin', 'met_hgb', 'platelets', 'white_blood_cell_count', 'carboxy_hgb', 'alanine_aminotransferase_(alt)', 'albumin', 'alkaline_phosphatase']

[First 5 rows]


,age,gender,race,cci9,cci10,anion_gap,base_excess,bicarb_(hco3),blood_urea_nitrogen_(bun),calcium,...,sofa_total,sofa_delta_24h,sofa_total_mod,sofa_delta_24h_mod,meld_score,aki_score,infection,sepsis,elapsed_icu,elapsed_hosp
2203-08-06 00:21:00,91,Male,BLACK/AFRICAN AMERICAN,4,0.0,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,6.43,0,0,0,0.0,0.0
2203-08-06 01:21:00,91,Male,BLACK/AFRICAN AMERICAN,4,0.0,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,6.43,0,0,0,0.0,0.0
2203-08-06 02:21:00,91,Male,BLACK/AFRICAN AMERICAN,4,0.0,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,6.43,0,0,0,0.0,0.0
2203-08-06 03:21:00,91,Male,BLACK/AFRICAN AMERICAN,4,0.0,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,6.43,0,0,0,0.0,0.0
2203-08-06 04:21:00,91,Male,BLACK/AFRICAN AMERICAN,4,0.0,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,6.43,0,0,0,0.0,1.0



[Column dtype summary]
float64    141
int64       15
object       7
bool         2
Name: count, dtype: int64


## 3. Match Supertables with CXR metadata

**Note**: Supertable filenames are **hadm_id** (hospital admission ID), not subject_id. Use MIMIC-IV admissions table for subject_id ↔ hadm_id mapping.

Flow: For each CXR (subject_id, StudyDate, StudyTime) → look up admissions for hadm_id → load hadm_id.pkl → match StudyDate exactly, StudyTime hour only → extract row and add dicom_id.

In [11]:
import pickle
import subprocess
import tempfile
from pathlib import Path
import pandas as pd
try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x  # fallback when tqdm not installed

METADATA_PATH = "/hpc/group/kamaleswaranlab/mimic_cxr/mimic_cxr_jpg/mimic-cxr-2.0.0-metadata.csv.gz"
ADMISSIONS_PATH = "/hpc/group/kamaleswaranlab/mimic_iv/mimic-iv-3.1-decompress/hosp/admissions.csv"
SUPERTABLES_DIR = Path("/hpc/group/kamaleswaranlab/mimic_iv/sepy_output/mimic-supertables/Supertables")
OUTPUT_PKL = "/work/gs285/cxr_supertable_matched.pkl"

def load_supertable_pkl(pkl_path):
    """Load pkl; if MedTVT-R1 numpy errors, use base env subprocess to convert to CSV and read"""
    try:
        with open(pkl_path, 'rb') as f:
            return pickle.load(f)
    except (RuntimeError, ImportError) as e:
        if "empty_like" in str(e) or "numpy" in str(e).lower():
            with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
                csv_path = tmp.name
            cmd = f'import pickle,sys,pandas as pd\npkl,csv=sys.argv[1],sys.argv[2]\nwith open(pkl,"rb") as f: d=pickle.load(f)\npd.DataFrame(d).to_csv(csv,index=True)\n'
            r = subprocess.run(["conda", "run", "-n", "base", "python", "-c", cmd, str(pkl_path), csv_path],
                               capture_output=True, text=True, timeout=120)
            if r.returncode != 0:
                raise RuntimeError(f"subprocess failed: {r.stderr}")
            return pd.read_csv(csv_path, index_col=0)
        raise

# 1. Load metadata and admissions
meta = pd.read_csv(METADATA_PATH)
admissions = pd.read_csv(ADMISSIONS_PATH, parse_dates=["admittime", "dischtime"])
print(f"Metadata: {len(meta):,} records CXR")
print(f"Admissions: {len(admissions):,} records")

# 2. Parse datetime for each CXR and match to hadm_id (CXR time must be within admittime-dischtime)
def studydatetime(row):
    sd, st = int(row["StudyDate"]), row["StudyTime"]
    y, m, d = sd // 10000, (sd % 10000) // 100, sd % 100
    h = int(st // 10000)
    return pd.Timestamp(year=y, month=m, day=d, hour=h, minute=0, second=0)

meta = meta.copy()
meta["_cxr_dt"] = meta.apply(studydatetime, axis=1)
meta["_date_int"] = meta["StudyDate"].astype(int)
meta["_hour"] = (meta["StudyTime"] // 10000).astype(int)

# Merge on subject_id, then filter CXR time within admission interval
meta_adm = meta.merge(admissions[["subject_id", "hadm_id", "admittime", "dischtime"]], on="subject_id")
meta_adm = meta_adm[(meta_adm["admittime"] <= meta_adm["_cxr_dt"]) & (meta_adm["dischtime"] >= meta_adm["_cxr_dt"])]
# If CXR matches multiple admissions, take first
cxr_df = meta_adm.drop_duplicates(subset=["dicom_id"], keep="first")[["subject_id", "StudyDate", "StudyTime", "dicom_id", "hadm_id", "_date_int", "_hour"]]
print(f"CXRs matched to hadm_id: {len(cxr_df):,} records")

# 3. Keep only records where hadm_id has existing pkl
existing_hadm = {int(f.stem) for f in SUPERTABLES_DIR.glob("*.pkl")}
cxr_df = cxr_df[cxr_df["hadm_id"].isin(existing_hadm)]
print(f"CXRs with existing supertable: {len(cxr_df):,} records")

Metadata: 377,110 CXR records
Admissions: 546,028 records
CXRs matched to hadm_id: 142,811 records
CXRs with existing supertable: 142,811 records


In [12]:
# 4. Match logic: load supertable by hadm_id, StudyDate exact, StudyTime hour only
def match_and_extract(cxr_with_hadm, supertables_dir, load_fn, max_hadm=None):
    """
    cxr_with_hadm: DataFrame with subject_id, dicom_id, hadm_id, _date_int, _hour
    """
    rows_list = []
    cxr_by_hadm = cxr_with_hadm.groupby("hadm_id")
    hadm_ids = list(cxr_by_hadm.groups.keys())
    if max_hadm:
        hadm_ids = hadm_ids[:max_hadm]
    try:
        iterator = tqdm(hadm_ids, desc="Matching")
    except NameError:
        iterator = hadm_ids
    
    for hadm_id in iterator:
        pkl_path = supertables_dir / f"{hadm_id}.pkl"
        if not pkl_path.exists():
            continue
        try:
            st = load_fn(pkl_path)
        except Exception as e:
            if hasattr(iterator, 'set_postfix_str'):
                iterator.set_postfix_str(f"load failed {hadm_id}")
            continue
        
        if not isinstance(st, pd.DataFrame) or st.empty:
            continue
            
        if not isinstance(st.index, pd.DatetimeIndex):
            st.index = pd.to_datetime(st.index)
        
        st = st.copy()
        st["_date_int"] = st.index.year * 10000 + st.index.month * 100 + st.index.day
        st["_hour"] = st.index.hour
        
        for _, row in cxr_by_hadm.get_group(hadm_id).iterrows():
            mask = (st["_date_int"] == row["_date_int"]) & (st["_hour"] == row["_hour"])
            matched = st.loc[mask]
            if len(matched) > 0:
                m = matched.iloc[0:1].copy().drop(columns=["_date_int", "_hour"])
                m["dicom_id"] = row["dicom_id"]
                m["subject_id"] = row["subject_id"]
                m["hadm_id"] = hadm_id
                m["supertable_datetime"] = matched.index[0]
                rows_list.append(m)
    
    if not rows_list:
        return pd.DataFrame()
    return pd.concat(rows_list, ignore_index=True)

# Small-scale test first
matched_df = match_and_extract(cxr_df, SUPERTABLES_DIR, load_supertable_pkl, max_hadm=200)
print(f"Test match result: {len(matched_df):,} rows")
# Note: 200 hadm_ids → 749 rows because each admission can have multiple CXRs; each CXR matches one supertable row
if len(matched_df) > 0:
    cols = ["subject_id", "dicom_id"] + [c for c in matched_df.columns if c not in ("subject_id", "dicom_id")][:5]
    display(matched_df[cols].head(10))
    # Save test result (MedTVT-R1 numpy causes to_pickle error, save CSV only)
    matched_df.to_csv("/work/gs285/matched_df_test.csv", index=False)
    print(f"Saved: matched_df_test.csv")

Matching: 100%|██████████| 200/200 [46:47<00:00, 14.04s/it] 


Test match result: 749 rows


,subject_id,dicom_id,age,gender,race,cci9,cci10
0,16207995,09adbdde-01be9f87-fdc68485-6290f966-676540b0,55,Female,WHITE,2.0,0.0
1,16207995,cecedb0d-f372d3a8-359e77b7-ebc7b648-14cb7c04,55,Female,WHITE,2.0,0.0
2,16788749,b8c23ba9-4d590858-3a5b2aee-769d1d89-fdc53733,54,Female,WHITE,0.0,0.0
3,19179793,4ced0006-01fc56cc-faabfb25-4a649653-ff88908b,59,Male,WHITE,6.0,0.0
4,19179793,61172f57-7746058d-7bbfa7de-ecb00d3a-5b77b8d2,59,Male,WHITE,6.0,0.0
5,19179793,40e3ef70-888a5e72-69228177-9f9c97ab-7a2f0857,59,Male,WHITE,6.0,0.0
6,19179793,8e5f2d69-2fbd5a24-10867cc8-0af74860-a0ae746a,59,Male,WHITE,6.0,0.0
7,11183547,32b9245e-c7ac6468-78b98376-b79966b1-1d27a4b2,79,Male,WHITE,0.0,0.0
8,11183547,78dee96f-d9758384-0789bc41-b35e7048-7fa2d241,79,Male,WHITE,0.0,0.0
9,11183547,fdc4737c-0c89f328-5bd278ec-3365310a-6824e746,79,Male,WHITE,0.0,0.0


In [14]:
# If cell 8 already run, execute this cell to save matched_df (no need to re-run matching)
# Note: MedTVT-R1 numpy causes to_pickle error, save CSV only; use pd.read_csv() to reload
try:
    csv_path = "/work/gs285/matched_df_test.csv"
    matched_df.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path} ({matched_df.shape[0]} rows × {matched_df.shape[1]} columns)")
except NameError:
    print("Please run the previous cell (match_and_extract) first")

Saved: /work/gs285/matched_df_test.csv (749 rows × 178 cols)


## check

In [16]:
# Using first row of matched_df, fetch raw data from metadata and supertable for validation
import pandas as pd
from pathlib import Path
import subprocess
import tempfile

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)

MATCHED_CSV = "/work/gs285/matched_df_test.csv"
METADATA_PATH = "/hpc/group/kamaleswaranlab/mimic_cxr/mimic_cxr_jpg/mimic-cxr-2.0.0-metadata.csv.gz"
SUPERTABLES_DIR = Path("/hpc/group/kamaleswaranlab/mimic_iv/sepy_output/mimic-supertables/Supertables")

# Read matched first row
matched = pd.read_csv(MATCHED_CSV)
row0 = matched.iloc[0]
dicom_id = row0["dicom_id"]
subject_id = row0["subject_id"]
hadm_id = int(row0["hadm_id"])
st_dt = row0["supertable_datetime"]

print("=" * 60)
print("[Key fields of matched first row]")
print("=" * 60)
print(f"dicom_id: {dicom_id}")
print(f"subject_id: {subject_id}")
print(f"hadm_id: {hadm_id}")
print(f"supertable_datetime: {st_dt}")
print("\n[Full matched first row data]")
display(matched.iloc[[0]])

# 1. Metadata raw data (one supertable row may match multiple CXRs, show all metadata)
same_supertable = matched[(matched["hadm_id"] == hadm_id) & (matched["supertable_datetime"] == st_dt)]
dicom_ids = same_supertable["dicom_id"].tolist()
meta = pd.read_csv(METADATA_PATH)
meta_rows = meta[meta["dicom_id"].isin(dicom_ids)]
print("\n" + "=" * 60)
print(f"[Metadata raw data] Same supertable row matches {len(dicom_ids)} CXRs")
print("=" * 60)
for i, did in enumerate(dicom_ids):
    print(f"\n--- CXR {i+1}: {did} ---")
    display(meta_rows[meta_rows["dicom_id"] == did])

# 2. Supertable raw data (load pkl via base env)
pkl_path = str(SUPERTABLES_DIR / f"{hadm_id}.pkl")
script = f'''import pickle,pandas as pd,sys
with open(r"{pkl_path}","rb") as f: st=pickle.load(f)
df=pd.DataFrame(st); df.index=pd.to_datetime(df.index)
pd.set_option("display.max_columns",None); pd.set_option("display.width",None)
dt=pd.Timestamp("{st_dt}")
m=(df.index.date==dt.date())&(df.index.hour==dt.hour)
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
print("Matched rows:",m.sum())
print(df.loc[m].to_string())
'''
with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as t:
    t.write(script)
    tpath = t.name
r = subprocess.run(["conda", "run", "-n", "base", "python", tpath], capture_output=True, text=True, timeout=60)
print("\n" + "=" * 60)
print(f"[Supertable raw data] hadm_id={hadm_id}, time={st_dt}")
print("=" * 60)
print(r.stdout if r.returncode == 0 else r.stderr)

[Key fields of matched first row]
dicom_id: 09adbdde-01be9f87-fdc68485-6290f966-676540b0
subject_id: 16207995
hadm_id: 20000471
supertable_datetime: 2171-04-12 11:39:00

[Full matched first row data]


,age,gender,race,cci9,cci10,anion_gap,base_excess,bicarb_(hco3),blood_urea_nitrogen_(bun),calcium,calcium_ionized,chloride,creatinine,gfr,glucose,magnesium,osmolarity,phosphorus,potassium,sodium,haptoglobin,hematocrit,hemoglobin,met_hgb,platelets,white_blood_cell_count,carboxy_hgb,alanine_aminotransferase_(alt),albumin,alkaline_phosphatase,ammonia,aspartate_aminotransferase_(ast),bilirubin_direct,bilirubin_total,fibrinogen,inr,lactate_dehydrogenase,lactic_acid,partial_prothrombin_time_(ptt),prealbumin,protein,prothrombin_time_(pt),thrombin_time,transferrin,amylase,lipase,b-type_natriuretic_peptide_(bnp),troponin,fio2,partial_pressure_of_carbon_dioxide_(paco2),partial_pressure_of_oxygen_(pao2),ph,saturation_of_oxygen_(sao2),d_dimer,hemoglobin_a1c,parathyroid_level,thyroid_stimulating_hormone_(tsh),lymphocyte,neutrophils,crp_high_sens,procalcitonin,erythrocyte_sedimentation_rate_(esr),c_diff,covid,mtp,pat_id,recorded_time,temperature,temproute,daily_weight_kg,height_cm,sbp_line,dbp_line,map_line,sbp_cuff,dbp_cuff,map_cuff,pulse,unassisted_resp_rate,spo2,glucose.1,o2_delivery_device_1,o2_delivery_device_2,o2_delivery_device_3,o2_delivery_device_4,cvp,end_tidal_co2,o2_flow_rate,temproute.1,o2_delivery_device_1.1,o2_delivery_device_2.1,o2_delivery_device_3.1,o2_delivery_device_4.1,in_or,proc_start,gcs_eye_score,gcs_verbal_score,gcs_motor_score,gcs_total_score,vent_status,vent_fio2,peep,vent_category,tidal_volume_observed,tidal_volume_spontaneous,vent_tidal_rate_set,vent_rate_set,icu,imc,ed,procedure,bed_unit,bed_type,icu_type,on_dialysis,history_of_dialysis,radiology_notes,clinical_notes,norepinephrine,epinephrine,dobutamine,dopamine,phenylephrine,vasopressin,norepinephrine_dose_unit,epinephrine_dose_unit,dobutamine_dose_unit,dopamine_dose_unit,phenylephrine_dose_unit,vasopressin_dose_unit,icd_procedure_desc,icd9_procedure_code,icd10_procedure_code,procedure_cpt_desc,procedure_cpt_code,best_map,pulse_pressure,n_to_l,s2f_vent_fio2,p2f_vent_fio2,norepinephrine_dose_weight,epinephrine_dose_weight,dobutamine_dose_weight,dopamine_dose_weight,phenylephrine_dose_weight,vasopressin_dose_weight,anion_gap.1,on_pressors,on_dobutamine,SIRS_resp,SIRS_cardio,SIRS_temp,SIRS_wbc,sirs_total,sirs_delta_24h,SOFA_coag,SOFA_renal,SOFA_hep,SOFA_neuro,SOFA_cardio,SOFA_cardio_mod,SOFA_resp,SOFA_resp_sa,sofa_total,sofa_delta_24h,sofa_total_mod,sofa_delta_24h_mod,meld_score,aki_score,infection,sepsis,elapsed_icu,elapsed_hosp,dicom_id,subject_id,hadm_id,supertable_datetime,carboxy_hgb.1
0,55,Female,WHITE,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75.0,161.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,Hematology/Oncology,ward,hematology/oncology,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,0,0,0,0,0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,3.0,0.0,3.0,7.497474,0,1,1,0.0,41.0,09adbdde-01be9f87-fdc68485-6290f966-676540b0,16207995,20000471,2171-04-12 11:39:00,NaN



[Metadata raw data] Same supertable row matches 2 CXRs

--- CXR 1: 09adbdde-01be9f87-fdc68485-6290f966-676540b0 ---


,dicom_id,subject_id,study_id,PerformedProcedureStepDescription,ViewPosition,Rows,Columns,StudyDate,StudyTime,ProcedureCodeSequence_CodeMeaning,ViewCodeSequence_CodeMeaning,PatientOrientationCodeSequence_CodeMeaning
234681,09adbdde-01be9f87-fdc68485-6290f966-676540b0,16207995,56435633,Performed Desc,PA,2021,1804,21710412,111510.0,CHEST (PA AND LAT),postero-anterior,Recumbent



--- CXR 2: cecedb0d-f372d3a8-359e77b7-ebc7b648-14cb7c04 ---


,dicom_id,subject_id,study_id,PerformedProcedureStepDescription,ViewPosition,Rows,Columns,StudyDate,StudyTime,ProcedureCodeSequence_CodeMeaning,ViewCodeSequence_CodeMeaning,PatientOrientationCodeSequence_CodeMeaning
234682,cecedb0d-f372d3a8-359e77b7-ebc7b648-14cb7c04,16207995,56435633,Performed Desc,LL,2021,1800,21710412,111510.0,CHEST (PA AND LAT),left lateral,Recumbent



[Supertable raw data] hadm_id=20000471, time=2171-04-12 11:39:00
Matched rows: 1
                     age  gender   race  cci9  cci10  anion_gap  base_excess  bicarb_(hco3)  blood_urea_nitrogen_(bun)  calcium  calcium_ionized  chloride  creatinine  gfr glucose  magnesium  osmolarity  phosphorus  potassium  sodium  haptoglobin  hematocrit  hemoglobin  met_hgb  platelets  white_blood_cell_count  carboxy_hgb  alanine_aminotransferase_(alt)  albumin  alkaline_phosphatase  ammonia  aspartate_aminotransferase_(ast)  bilirubin_direct  bilirubin_total  fibrinogen  inr  lactate_dehydrogenase  lactic_acid  partial_prothrombin_time_(ptt)  prealbumin  protein  prothrombin_time_(pt)  thrombin_time  transferrin  amylase  lipase  b-type_natriuretic_peptide_(bnp)  troponin  fio2  partial_pressure_of_carbon_dioxide_(paco2)  partial_pressure_of_oxygen_(pao2)  ph  saturation_of_oxygen_(sao2)  d_dimer  hemoglobin_a1c  parathyroid_level  thyroid_stimulating_hormone_(tsh)  lymphocyte  neutrophils  crp_high

In [ ]:
# 5. Full run and save
matched_full = match_and_extract(cxr_df, SUPERTABLES_DIR, load_supertable_pkl)
print(f"Total matches: {len(matched_full):,} rows")

# MedTVT-R1 numpy causes to_pickle error; save CSV first, then convert to pkl via base env
OUTPUT_CSV = "/work/gs285/cxr_supertable_matched.csv"
matched_full.to_csv(OUTPUT_CSV, index=False)
print(f"Saved CSV: {OUTPUT_CSV}")
subprocess.run(["conda", "run", "-n", "base", "python", "-c",
    f"import pandas as pd; pd.read_csv('{OUTPUT_CSV}').to_pickle('{OUTPUT_PKL}'); print('Saved pkl:', '{OUTPUT_PKL}')"],
    check=True, capture_output=True, text=True)
print(f"Saved pkl: {OUTPUT_PKL}")
print(f"DataFrame Columns: {len(matched_full.columns)}")

匹配:   0%|          | 54/38858 [11:10<133:52:41, 12.42s/it]

### Option A: Background run ( survives Cursor close )
Run the cell below to start the full match in background. Process continues even after you close the IDE. Monitor: `tail -f /work/gs285/run_full_match.log`

In [4]:
# Run full match in BACKGROUND (continues after closing Cursor)
# Execute this cell, then you can safely close - check log: tail -f /work/gs285/run_full_match.log
import subprocess
import sys
proc = subprocess.Popen(
    ["nohup", sys.executable, "/work/gs285/run_full_match.py"],
    stdout=open("/work/gs285/run_full_match.log", "w"),
    stderr=subprocess.STDOUT,
    cwd="/work/gs285",
    start_new_session=True
)
print(f"Started PID {proc.pid}. Runs in background.")
print("Monitor: tail -f /work/gs285/run_full_match.log")

Started PID 3759664. Runs in background.
Monitor: tail -f /work/gs285/run_full_match.log
